# Module 7 lab · Gradients and your own autograd

The code from every lesson of [this module](https://aicourses.dev/research/gradients), in order. Read a lesson, then run its cells here: each lesson builds on the one before, so run the cells from the top the first time.

**First time?** Read the [one-time setup](https://aicourses.dev/research/labs): a Google account, your own copy, and what to expect.

The lessons that explain this code are part of [AI Courses Pro](https://aicourses.dev/#pricing); the module’s first lesson is free.

Under each cell, *The lesson shows* is what it should print. Your last digits can differ a little.

**Keep your work:** File → Save a copy in Drive. Colab does not keep changes to this copy.

## 7.2 Derivatives: the slope of a curve at one point

Module 3 measured slopes by nudging an input. A derivative is that nudge made tiny, so it gives the slope at one exact point. You check every derivative by nudging.

[Read the lesson](https://aicourses.dev/research/gradients/derivatives)

### Check every derivative by nudging

From now on, every derivative in this course gets checked the same way: compute the nudge slope in code and compare. Nudging both ways, to $x + h$ and $x - h$, and dividing by $2h$ is more accurate than nudging one way, because the errors on each side mostly cancel. At 3 with $h = 0.1$: $(3.1^2 - 2.9^2) \div 0.2 = (9.61 - 8.41) \div 0.2 = 6$, exactly, where one way gave 6.1.

In [ ]:
def f(x):
    return x ** 2

def nudge_slope(f, x, h):
    return (f(x + h) - f(x)) / h      # rise over run

for h in [1, 0.1, 0.01, 0.001]:
    print(h, nudge_slope(f, 3, h))
# 1 7.0
# 0.1 6.100000000000012
# 0.01 6.009999999999849
# 0.001 6.000999999999479

def central_slope(f, x, h=1e-5):
    return (f(x + h) - f(x - h)) / (2 * h)   # nudge both ways

print(central_slope(f, 3))           # 6.000000000039306

The tiny tails such as 6.100000000000012 come from rounding: a computer stores most decimals only approximately. That is also why you must not make $h$ too small. A Python float keeps about 16 significant digits, and $f(3 + h)$ and $f(3)$ agree in almost all of them, so their difference keeps only the noisy last few:

In [ ]:
for h in [1e-6, 1e-12, 1e-15]:
    print(h, nudge_slope(f, 3, h))
# 1e-06 6.000001000927568
# 1e-12 6.000533403494046
# 1e-15 5.329070518200751

### A slope you will need: the log

Cross-entropy, the loss for choices from lesson 6.7, contains a log, so measure its slope too. `math.log` is the natural log from Module 3:

In [ ]:
import math

print(central_slope(math.log, 2))    # 0.5000000000088267
print(central_slope(math.log, 4))    # 0.24999999999053554

## 7.3 The few derivative rules you need

You only need a handful of rules: for powers, sums, products, e to the x, and logs. You check each rule by nudging, so you can trust it in the lessons that follow.

[Read the lesson](https://aicourses.dev/research/gradients/derivative-rules)

### Check every rule in code

`lambda x: 3 * x**2` is a one-line function, handy when you need many small ones. The `check` function prints the nudged slope beside the rule’s slope; in its f-string, `:9` pads the name to nine characters so the columns line up, and `:.6f` shows six decimals:

In [ ]:
import math

def central_slope(f, x, h=1e-5):
    return (f(x + h) - f(x - h)) / (2 * h)   # nudge both ways

def check(name, f, rule, x):
    nudged, exact = central_slope(f, x), rule(x)
    print(f"{name:9} nudge {nudged:.6f}  rule {exact:.6f}")

def sigmoid(z):
    return 1 / (1 + math.exp(-z))

check("x**3", lambda x: x**3, lambda x: 3 * x**2, 2)
check("3x²+2x", lambda x: 3*x**2 + 2*x, lambda x: 6*x + 2, 1)
check("e**x", math.exp, math.exp, 1)
check("ln x", math.log, lambda x: 1 / x, 2)
check("x·e**x", lambda x: x * math.exp(x),
      lambda x: math.exp(x) + x * math.exp(x), 1)
check("tanh", math.tanh, lambda z: 1 - math.tanh(z) ** 2, 0.5)
check("sigmoid", sigmoid,
      lambda z: sigmoid(z) * (1 - sigmoid(z)), 0.5)

*The lesson shows:*

```text
x**3      nudge 12.000000  rule 12.000000
3x²+2x    nudge 8.000000  rule 8.000000
e**x      nudge 2.718282  rule 2.718282
ln x      nudge 0.500000  rule 0.500000
x·e**x    nudge 5.436564  rule 5.436564
tanh      nudge 0.786448  rule 0.786448
sigmoid   nudge 0.235004  rule 0.235004
```

## 7.4 Many knobs at once: partial derivatives and the gradient

A model has millions of knobs. The gradient lists how the loss moves for each one, and it points uphill, so training steps the other way.

[Read the lesson](https://aicourses.dev/research/gradients/partial-derivatives)

### Check by nudging each knob

Nudge one knob both ways while the other stays put, exactly as in lesson 7.2:

In [ ]:
def loss(w, b):
    prediction = w * 2 + b        # the input is 2
    return (prediction - 3) ** 2  # the right answer is 3

h = 1e-5
w, b = 0.5, 1.0
print(loss(w, b))                                     # 1.0
dL_dw = (loss(w + h, b) - loss(w - h, b)) / (2 * h)   # b held still
dL_db = (loss(w, b + h) - loss(w, b - h)) / (2 * h)   # w held still
print(round(dL_dw, 6), round(dL_db, 6))               # -4.0 -2.0

## 7.5 Gradient descent: walking downhill

Training walks downhill on the loss: take a small step against the gradient, then repeat. The size of each step is called the learning rate. You learn the rule on a simple bowl-shaped curve with two knobs before a model with billions.

[Read the lesson](https://aicourses.dev/research/gradients/derivatives-and-gradient-descent)

### Walk down a bowl by hand

Two steps and the loss has fallen from 7 to 1.7. A loop does the rest:

In [ ]:
def loss(w1, w2):
    return w1 ** 2 + 3 * w2 ** 2      # a bowl, lowest at (0, 0)

def gradient(w1, w2):
    return 2 * w1, 6 * w2             # from the power rule

w1, w2 = 2.0, 1.0                     # where we start
lr = 0.1                              # the learning rate
for step in range(1, 21):
    g1, g2 = gradient(w1, w2)
    w1, w2 = w1 - lr * g1, w2 - lr * g2   # step against the gradient
    if step in (1, 2, 10, 20):
        print(step, round(w1, 4), round(w2, 4), round(loss(w1, w2), 4))
# 1 1.6 0.4 3.04
# 2 1.28 0.16 1.7152
# 10 0.2147 0.0001 0.0461
# 20 0.0231 0.0 0.0005

### A real model: fitting a line

The bowl was made up. Here is a real use. Four deliveries took 14, 21, 24 and 31 minutes for 1, 2, 3 and 4 kilometres. Fit the line $\text{minutes} = w \times \text{km} + b$ by making its mean squared error small. You do not need a formula for the gradient: measure it by nudging, as in lesson 7.4.

In [ ]:
km = [1, 2, 3, 4]                     # distance of four deliveries
minutes = [14, 21, 24, 31]            # how long each took

def loss(w, b):                       # mean squared error of the line
    errors = [w * d + b - t for d, t in zip(km, minutes)]
    return sum(e * e for e in errors) / len(errors)

def nudged_gradient(w, b, h=1e-5):
    dw = (loss(w + h, b) - loss(w - h, b)) / (2 * h)
    db = (loss(w, b + h) - loss(w, b - h)) / (2 * h)
    return dw, db

w, b, lr = 0.0, 0.0, 0.05
for step in range(1, 2001):
    dw, db = nudged_gradient(w, b)
    w, b = w - lr * dw, b - lr * db
    if step in (1, 100, 2000):
        print(step, round(w, 3), round(b, 3), round(loss(w, b), 3))
# 1 6.3 2.25 22.063
# 100 5.883 7.579 1.137
# 2000 5.4 9.0 0.8

## 7.6 The chain rule, drawn as a graph

When one calculation feeds the next, the chain rule says how a nudge at the start moves the end: multiply the slopes along the way. Draw the calculation as boxes and arrows, called a computation graph, and backpropagation is this rule run from the end back to the start.

[Read the lesson](https://aicourses.dev/research/gradients/chain-rule-graph)

### The backward pass in code

In [ ]:
# forward pass, one small step at a time
w, x, b, target = 0.5, 2.0, 1.0, 3.0
p = w * x              # 1.0
s = p + b              # 2.0
d = s - target         # -1.0
L = d ** 2             # 1.0

# backward pass: start at the end, where dL/dL = 1
dL_dL = 1.0
dL_dd = 2 * d * dL_dL  # square: local slope 2d
dL_ds = 1 * dL_dd      # subtract: local slope 1
dL_dp = 1 * dL_ds      # add: copies the gradient
dL_db = 1 * dL_ds
dL_dw = x * dL_dp      # multiply: times the other input
dL_dx = w * dL_dp
print(L, dL_dw, dL_db, dL_dx)   # 1.0 -4.0 -2.0 -1.0

## 7.7 Build micrograd: a tiny engine that finds gradients for you

Autograd is code that records each calculation, then runs the chain rule backwards to get every gradient. You write a small one, about 100 lines, and test its answers against slopes you measure by nudging.

[Read the lesson](https://aicourses.dev/research/gradients/micrograd)

### A number that remembers

Python lets a class decide what `+` means for it by defining a method called `__add__`. So when you write `a + b` with two `Value`s, the class builds a new `Value` that remembers `a` and `b` as its children and stores how to send a gradient back to them:

In [ ]:
class Value:
    """A number that remembers where it came from."""

    def __init__(self, data, children=()):
        self.data = data
        self.grad = 0.0                # dL/d(this), set by backward
        self._children = children      # the Values that made this one
        self._backward = lambda: None  # sends grad to the children

    def __repr__(self):
        return f"Value(data={self.data}, grad={self.grad})"

    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data + other.data, (self, other))

        def _backward():               # add copies the gradient
            self.grad += out.grad
            other.grad += out.grad
        out._backward = _backward
        return out

### Multiply and power

Python also lets you attach a method to a class after defining it, which keeps each piece short:

In [ ]:
def mul(self, other):
    other = other if isinstance(other, Value) else Value(other)
    out = Value(self.data * other.data, (self, other))

    def _backward():                   # each gets the other input
        self.grad += other.data * out.grad
        other.grad += self.data * out.grad
    out._backward = _backward
    return out

def power(self, n):                    # n is a plain number
    out = Value(self.data ** n, (self,))

    def _backward():                   # power rule: n times x^(n-1)
        self.grad += n * self.data ** (n - 1) * out.grad
    out._backward = _backward
    return out

Value.__mul__ = mul                    # attach them to the class
Value.__pow__ = power

### The bends

The two activations from Module 6, each with the slope from lesson 7.3:

In [ ]:
import math

def tanh(self):
    t = math.tanh(self.data)
    out = Value(t, (self,))

    def _backward():                   # slope of tanh: 1 - tanh²
        self.grad += (1 - t ** 2) * out.grad
    out._backward = _backward
    return out

def relu(self):
    out = Value(max(0.0, self.data), (self,))

    def _backward():                   # slope 1 if positive, else 0
        self.grad += (1.0 if out.data > 0 else 0.0) * out.grad
    out._backward = _backward
    return out

Value.tanh = tanh
Value.relu = relu

### backward(): the right order

A value may pass its gradient back only once it has collected everything coming to it, from every value that used it. So `backward` first lists the values in **topological order**, an order in which every value comes after everything it was made from, and then walks that list in reverse, from the loss back to the inputs:

In [ ]:
def backward(self):
    order, seen = [], set()

    def visit(v):                      # children before parents
        if v not in seen:
            seen.add(v)
            for child in v._children:
                visit(child)
            order.append(v)
    visit(self)

    self.grad = 1.0                    # dL/dL = 1
    for v in reversed(order):          # from the loss backwards
        v._backward()

Value.backward = backward

Last, a few one-liners so that subtraction, negation, division and numbers on the left all work. Each is built from the pieces above, so none needs its own `_backward`:

In [ ]:
Value.__radd__ = lambda self, other: self + other   # 2 + v
Value.__rmul__ = lambda self, other: self * other   # 2 * v
Value.__neg__ = lambda self: self * -1              # -v
Value.__sub__ = lambda self, other: self + (-other)
Value.__rsub__ = lambda self, other: other + (-self)
Value.__truediv__ = lambda self, other: self * other ** -1

### Test it

First the neuron from lesson 7.6, where you know every answer:

In [ ]:
w, x, b = Value(0.5), Value(2.0), Value(1.0)
L = (w * x + b - 3) ** 2       # the neuron from lesson 7.6
L.backward()
print(L.data, w.grad, b.grad, x.grad)   # 1.0 -4.0 -2.0 -1.0

The same −4, −2 and −1 you worked out by hand. Now a test you could not easily do by hand: a made-up expression using every operation, checked against nudging each input:

In [ ]:
def expr(a, b):                        # a made-up test expression
    c = a * b + b ** 3
    return (c / 2).tanh() + 2 * a.relu() - b

def nudged(f, a, b, which, h=1e-6):    # slope by nudging one input
    if which == "a":
        up, down = f(Value(a + h), Value(b)), f(Value(a - h), Value(b))
    else:
        up, down = f(Value(a), Value(b + h)), f(Value(a), Value(b - h))
    return (up.data - down.data) / (2 * h)

a, b = Value(0.7), Value(-1.3)
expr(a, b).backward()
print(a.grad, nudged(expr, 0.7, -1.3, "a"))
# 1.893436582991094 1.8934365830780564
print(b.grad, nudged(expr, 0.7, -1.3, "b"))
# -0.5270223721989326 -0.5270223720765799

## 7.8 Training a tiny network on your engine

Module 6’s network had weights you set by hand. Now your engine finds them. This kind of network, layers of neurons one after another, is called a multilayer perceptron, or MLP. It learns XOR, then a small made-up dataset of two interlocking half-moons.

[Read the lesson](https://aicourses.dev/research/gradients/neurons-and-layers)

### Neurons, layers and an MLP, as code

A neuron holds a list of weights and a bias, each a `Value`, so the engine can find their gradients. The weights start as random numbers between −1 and 1, from Python’s `random.uniform`, and the bias at 0; in `for _ in range(n_inputs)`, `_` names a counter you never use. A class can define `__call__` to be used like a function, so `neuron(x)` runs the neuron on the inputs `x`:

In [ ]:
import random

class Neuron:
    def __init__(self, n_inputs, bend=True):
        self.w = [Value(random.uniform(-1, 1))
                  for _ in range(n_inputs)]
        self.b = Value(0.0)
        self.bend = bend               # the output layer has no bend

    def __call__(self, x):             # lets you write neuron(x)
        z = self.b
        for wi, xi in zip(self.w, x):
            z = z + wi * xi            # weighted sum plus bias
        return z.tanh() if self.bend else z

    def parameters(self):
        return self.w + [self.b]

A layer is a list of neurons that all read the same inputs, as in lesson 6.4:

In [ ]:
class Layer:
    def __init__(self, n_inputs, n_neurons, bend=True):
        self.neurons = [Neuron(n_inputs, bend)
                        for _ in range(n_neurons)]

    def __call__(self, x):
        out = [n(x) for n in self.neurons]
        return out[0] if len(out) == 1 else out

    def parameters(self):
        return [p for n in self.neurons for p in n.parameters()]

An MLP is a list of layers, each reading the last one’s outputs. Every layer bends with tanh except the last, whose output is a raw score, as in Module 6:

In [ ]:
class MLP:
    def __init__(self, n_inputs, sizes):   # e.g. MLP(2, [4, 1])
        dims = [n_inputs] + sizes
        last = len(sizes) - 1
        self.layers = [Layer(dims[i], dims[i + 1], bend=(i != last))
                       for i in range(len(sizes))]

    def __call__(self, x):
        for layer in self.layers:      # the forward pass
            x = layer(x)
        return x

    def parameters(self):
        return [p for l in self.layers for p in l.parameters()]

### The training loop

Training repeats four moves: run the forward pass and compute the loss, set every gradient back to 0, run the backward pass, and nudge every knob against its gradient. That is gradient descent from lesson 7.5, with the gradients now coming from your engine instead of from nudging.

In [ ]:
random.seed(0)                     # the same random start every run
X = [[0, 0], [0, 1], [1, 0], [1, 1]]   # the four XOR cases
y = [0, 1, 1, 0]
model = MLP(2, [4, 1])        # 2 inputs, 4 hidden neurons, 1 output

for step in range(501):
    preds = [model(x) for x in X]                       # forward
    loss = sum((p - t) ** 2 for p, t in zip(preds, y)) / len(y)
    for p in model.parameters():
        p.grad = 0.0                   # forget last step’s gradients
    loss.backward()                    # backward
    for p in model.parameters():
        p.data -= 0.1 * p.grad         # downhill, learning rate 0.1
    if step % 100 == 0:
        print(step, round(loss.data, 4))
print([round(model(x).data, 2) for x in X])

*The lesson shows:*

```text
0 0.6766
100 0.184
200 0.0582
300 0.0048
400 0.0002
500 0.0
[0.0, 1.0, 1.0, 0.0]
```

### A harder shape: two half-moons

XOR has four points. Here is something closer to real data: 100 points in two interlocking half-moons, one labelled +1 and one −1, with a little random wobble so the moons are fuzzy. No straight line separates them, so the network has to learn a curved boundary. `random.gauss(0, noise)` gives a random number centred on 0, mostly within about `noise` of it.

In [ ]:
import math

def make_moons(n, noise=0.1):
    points, labels = [], []
    for i in range(n):
        t = math.pi * random.random()        # a spot on a half-circle
        if i % 2 == 0:                       # the top moon: label +1
            x, y, label = math.cos(t), math.sin(t), 1
        else:                            # the bottom moon: label -1
            x, y, label = 1 - math.cos(t), 0.5 - math.sin(t), -1
        x += random.gauss(0, noise)          # a little random wobble
        y += random.gauss(0, noise)
        points.append([x, y])
        labels.append(label)
    return points, labels

Train a bigger MLP, with two hidden layers of 8 neurons, 105 knobs in all, the same way. A point counts as right when the output has the same sign as its label, and `sum` of True and False counts the Trues:

In [ ]:
random.seed(0)
X, y = make_moons(100)
model = MLP(2, [8, 8, 1])
for step in range(301):
    preds = [model(x) for x in X]
    loss = sum((p - t) ** 2 for p, t in zip(preds, y)) / len(y)
    for p in model.parameters():
        p.grad = 0.0
    loss.backward()
    for p in model.parameters():
        p.data -= 0.2 * p.grad
    if step % 100 == 0:
        right = sum((p.data > 0) == (t > 0) for p, t in zip(preds, y))
        print(step, round(loss.data, 4), f"{right} of 100 right")

*The lesson shows:*

```text
0 0.435 87 of 100 right
100 0.1057 98 of 100 right
200 0.0585 100 of 100 right
300 0.0385 100 of 100 right
```

## 7.9 Project: micrograd, tested

Proves your engine right the way engineers prove code right: with tests against a trusted reference.

[Read the lesson](https://aicourses.dev/research/gradients/project-micrograd-tested)

**This is the project.** These cells are starting points from the lesson; the steps between them are yours to write. A cell that uses a name from one of your steps stops with a NameError until you have written that step.

### Steps

2. **Write the gradient checks.** PyTorch’s version of your `Value` is a tensor made with `requires_grad=True`: it records its calculations and has `.backward()` and `.grad` just like yours. Module 8 opens it up; here you only use it as a reference. `dtype=torch.float64` makes it use the same precision as a Python float. In a new cell:

In [ ]:
import torch

def grads(f, a, b):
    """Gradients of f at (a, b): engine, PyTorch and nudging."""
    va, vb = Value(a), Value(b)
    f(va, vb).backward()
    ta = torch.tensor(a, dtype=torch.float64, requires_grad=True)
    tb = torch.tensor(b, dtype=torch.float64, requires_grad=True)
    f(ta, tb).backward()
    h = 1e-6
    na = (f(Value(a + h), Value(b)).data
          - f(Value(a - h), Value(b)).data) / (2 * h)
    nb = (f(Value(a), Value(b + h)).data
          - f(Value(a), Value(b - h)).data) / (2 * h)
    mine = (va.grad, vb.grad)
    ref = (ta.grad.item(), tb.grad.item())
    return mine, ref, (na, nb)

def check(f, a, b):
    mine, ref, nudged = grads(f, a, b)
    for m, r, n in zip(mine, ref, nudged):
        assert abs(m - r) < 1e-9, f"engine {m} vs PyTorch {r}"
        assert abs(m - n) < 1e-5, f"engine {m} vs nudging {n}"

3. **Write one test per likely bug**, then a loop that runs them all:

In [ ]:
def test_add_and_mul():
    check(lambda a, b: a * b + a, 2.0, -3.0)

def test_power_and_divide():
    check(lambda a, b: a ** 3 / b - b, 1.5, 4.0)

def test_bends():
    check(lambda a, b: (a * b).tanh() + b.relu(), 0.3, 0.8)

def test_value_used_twice():
    check(lambda a, b: a * a + a * b, -1.2, 0.5)

def test_neuron_loss():
    check(lambda w, b: (w * 2 + b - 3) ** 2, 0.5, 1.0)

for name, test in list(globals().items()):
    if name.startswith("test_"):
        test()
        print("passed", name)

5. **Work a two-layer gradient by hand.** The network has one neuron per layer: $h = \tanh(w_1 x + b_1)$, then $\text{out} = w_2 h + b_2$, and the loss $L = (\text{out} - y)^2$. Use $x = 2$, $y = 1$, $w_1 = 0.5$, $b_1 = 0$, $w_2 = -1$, $b_2 = 0$. On paper, run the forward pass, then the backward pass box by box as in lesson 7.6, using the tanh slope $1 - h^2$ from lesson 7.3. Write down all four gradients. Then check them:

In [ ]:
x, t = 2.0, 1.0                              # one input, its target
w1, b1 = Value(0.5), Value(0.0)              # hidden neuron
w2, b2 = Value(-1.0), Value(0.0)             # output neuron

h = (w1 * x + b1).tanh()                     # layer 1, then the bend
out = w2 * h + b2                            # layer 2
L = (out - t) ** 2
L.backward()
for name, v in [("w1", w1), ("b1", b1), ("w2", w2), ("b2", b2)]:
    print(name, round(v.grad, 4))

6. **Train on the moons and plot the boundary.** Train `MLP(2, [8, 8, 1])` on `make_moons(100)` with `random.seed(0)`, as in lesson 7.8. Then shade the plane by the network’s output:

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

xs = np.linspace(-1.5, 2.5, 60)              # a grid over the plane
ys = np.linspace(-1.0, 1.5, 60)
grid = [[model([float(a), float(b)]).data for a in xs] for b in ys]

plt.contourf(xs, ys, grid, levels=[-1e9, 0, 1e9],
             colors=["#9cc3e6", "#f4c27a"])  # blue: -1, orange: +1
plt.scatter([p[0] for p in X], [p[1] for p in X], c=y,
            cmap="coolwarm", edgecolors="black")
plt.title("micrograd MLP: decision boundary")
plt.savefig("moons_boundary.png")